# Valuation-to-Option Pricing Lab

This notebook keeps two valuation tracks separate:

1. **Analyst-scenario discounted expected payoff:** five terminal equity-value scenarios with subjective weights.
2. **CRR risk-neutral lattice value:** a no-arbitrage option-pricing model built through repeated two-state replication.

The five analyst scenarios are not an expanded binomial tree and are not risk-neutral probabilities. They answer a different question: *Given these valuation beliefs and weights, what is the discounted expected option payoff?*

**Extensions in this version** (all additive; the CRR function, validation assertions, workflow, requirements and `docs/index.html` are unchanged):

- Distribution diagnostics: skewness, excess kurtosis, IQR, Tukey outliers, dispersion.
- An optional explicit DCF build from `data/dcf_inputs.csv`, reconciled against the reported `dcf_value`.
- Weight-sensitivity of the five-state result (disclosed, equal and data-implied weights).
- A distribution-based belief value: 20,000 terminal prices drawn from a kernel density fitted to analyst targets.
- A Black-Scholes benchmark for the CRR lattice, mispricing metrics against bid/mid/ask, and sensitivity tables.
- Maturity-alignment and model-risk flags, a source-documentation audit, and a dated input snapshot.
- A supplementary dashboard at `docs/analysis.html` (convergence chart, volatility card, model-risk badges).

| Section | Purpose |
|---|---|
| 1 | Load and validate inputs |
| 2 | Equity valuation, DCF build, analyst distribution diagnostics |
| 3–4 | Five analyst states, payoff decomposition, weight sensitivity |
| 5 | Distribution-based (KDE) belief value |
| 6–7 | CRR lattice, theoretical validation, Black-Scholes benchmark, mispricing |
| 8 | Sensitivity analysis |
| 9 | Maturity alignment, source audit, model-risk flags |
| 10 | Export dashboard data |
| 11 | Decision memo |

## 1. Load and validate student inputs

In [ ]:
from __future__ import annotations
from pathlib import Path
from math import erf, sqrt
import json, numpy as np, pandas as pd
ROOT=Path.cwd(); DATA=ROOT/'data'; DOCS=ROOT/'docs'; DOCS.mkdir(exist_ok=True)
inputs=pd.read_csv(DATA/'company_inputs.csv')
if len(inputs)!=1: raise ValueError('company_inputs.csv must contain exactly one firm row.')
M=inputs.iloc[0].to_dict()
for k in ['spot','strike','maturity_years','risk_free_rate','dividend_yield','volatility','market_bid','market_ask','dcf_value','comps_value']: M[k]=float(M[k])
M['lattice_steps']=int(M['lattice_steps']); M['market_mid']=(M['market_bid']+M['market_ask'])/2
M['option_type']=str(M['option_type']).lower(); M['exercise_style']=str(M['exercise_style']).lower()
if not 0<=M['risk_free_rate']<=1: raise ValueError('Enter risk_free_rate in decimal form, such as 0.045.')
if not 0<M['volatility']<=3: raise ValueError('Enter volatility in decimal form, such as 0.28.')
if M['market_bid']>M['market_ask']: raise ValueError('market_bid cannot exceed market_ask.')
if M['option_type'] not in ['call','put']: raise ValueError('option_type must be call or put.')
if M['exercise_style'] not in ['european','american']: raise ValueError('exercise_style must be european or american.')
targets=pd.read_csv(DATA/'analyst_targets.csv')
if len(targets)<5: raise ValueError('At least five analyst targets are required.')
if not targets.target.gt(0).all(): raise ValueError('Analyst targets must be positive.')
if targets.horizon_months.nunique()!=1: raise ValueError('All analyst targets must use one horizon.')
if targets.analyst.duplicated().any(): raise ValueError('Each analyst report must appear only once.')
M

## 2. Equity valuation and analyst distribution

In [ ]:
stats={'count':int(len(targets)),'mean':float(targets.target.mean()),'median':float(targets.target.median()),'std':float(targets.target.std(ddof=1)),'min':float(targets.target.min()),'max':float(targets.target.max())}
valuation={'Current market':M['spot'],'Comparable companies':M['comps_value'],'Student DCF':M['dcf_value'],'Analyst median':stats['median']}
pd.Series(valuation)

### 2a. Valuation upside versus the current price
Each estimate is expressed as implied upside or downside relative to spot, so the valuation evidence can be compared on one scale.

In [ ]:
valuation_table=pd.DataFrame({'value_per_share':valuation}).assign(
    upside_vs_spot=lambda d: d.value_per_share/M['spot']-1)
valuation_table.style.format({'value_per_share':'${:,.2f}','upside_vs_spot':'{:+.1%}'})

### 2b. Explicit DCF build (optional input file)
If `data/dcf_inputs.csv` exists, the notebook projects five years of free cash flow to the firm, discounts it at WACC, adds a Gordon-growth terminal value, bridges from enterprise value to equity value per share, and reconciles the result with the reported `dcf_value`. Replace the template assumptions with your own researched values.

In [ ]:
def build_dcf(params: dict[str, float]) -> dict:
    """Five-year FCFF projection with a Gordon-growth terminal value."""
    years=range(1,6); revenue=params['base_revenue']; rows=[]
    wacc, g=params['wacc'], params['terminal_growth']
    if not wacc>g: raise ValueError('WACC must exceed terminal growth.')
    for t in years:
        revenue*=1+params[f'revenue_growth_y{t}']
        ebit=revenue*params[f'ebit_margin_y{t}']
        nopat=ebit*(1-params['tax_rate'])
        fcff=nopat*(1-params['reinvestment_rate'])
        df=1/(1+wacc)**t
        rows.append({'year':t,'revenue':revenue,'ebit':ebit,'nopat':nopat,'fcff':fcff,'discount_factor':df,'pv_fcff':fcff*df})
    proj=pd.DataFrame(rows)
    terminal_value=proj.fcff.iloc[-1]*(1+g)/(wacc-g)
    pv_terminal=terminal_value*proj.discount_factor.iloc[-1]
    enterprise_value=proj.pv_fcff.sum()+pv_terminal
    equity_value=enterprise_value-params['net_debt']
    per_share=equity_value/params['diluted_shares']
    return {'projection':proj,'terminal_value':terminal_value,'pv_terminal':pv_terminal,'enterprise_value':enterprise_value,
            'equity_value':equity_value,'value_per_share':per_share,'terminal_share_of_ev':pv_terminal/enterprise_value,'wacc':wacc,'terminal_growth':g}

dcf_path=DATA/'dcf_inputs.csv'; dcf=None
if dcf_path.exists():
    dcf_params=pd.read_csv(dcf_path).set_index('parameter')['value'].astype(float).to_dict()
    dcf=build_dcf(dcf_params)
    dcf['reported_dcf_value']=M['dcf_value']; dcf['reconciliation_gap']=dcf['value_per_share']-M['dcf_value']
    wacc_grid=[dcf_params['wacc']+x for x in (-.01,-.005,0,.005,.01)]; g_grid=[dcf_params['terminal_growth']+x for x in (-.005,0,.005)]
    dcf_sensitivity=pd.DataFrame({f'g={g:.1%}':[build_dcf({**dcf_params,'wacc':w,'terminal_growth':g})['value_per_share'] for w in wacc_grid] for g in g_grid},index=[f'WACC={w:.1%}' for w in wacc_grid])
    print(f"Built DCF value per share: ${dcf['value_per_share']:,.2f} (reported dcf_value ${M['dcf_value']:,.2f}, gap {dcf['reconciliation_gap']:+.2f})")
    print(f"Terminal value share of enterprise value: {dcf['terminal_share_of_ev']:.1%}")
    display(dcf['projection'].round(2)); display(dcf_sensitivity.round(2))
else:
    print('data/dcf_inputs.csv not found; the reported dcf_value is used without an explicit build.')

### 2c. Analyst distribution diagnostics
Central tendency, dispersion, shape and outliers. Outliers use Tukey fences (1.5 × IQR beyond the quartiles); they are flagged, not removed.

In [ ]:
def distribution_diagnostics(x: pd.Series, spot: float) -> dict:
    q1,q3=float(x.quantile(.25)),float(x.quantile(.75)); iqr=q3-q1
    lo_f,hi_f=q1-1.5*iqr,q3+1.5*iqr
    return {'skewness':float(x.skew()),'excess_kurtosis':float(x.kurt()),'q1':q1,'q3':q3,'iqr':iqr,
            'coefficient_of_variation':float(x.std(ddof=1)/x.mean()),
            'lower_fence':lo_f,'upper_fence':hi_f,
            'tukey_outliers':[float(v) for v in x[(x<lo_f)|(x>hi_f)]],
            'lowest_target':float(x.min()),'highest_target':float(x.max()),
            'share_above_spot':float((x>spot).mean()),'share_above_strike':float((x>M['strike']).mean()),
            'publication_span_days':int((pd.to_datetime(targets.publication_date).max()-pd.to_datetime(targets.publication_date).min()).days)}
diagnostics=distribution_diagnostics(targets.target,M['spot'])
pd.Series(diagnostics)

## 3. Five analyst-informed valuation states

The state prices are reproducible quantiles of the analyst-target distribution. The weights are disclosed subjective scenario masses, not risk-neutral probabilities.

- Severe bear: 5th percentile, 10% weight
- Bear: 25th percentile, 20% weight
- Base: 50th percentile, 40% weight
- Bull: 75th percentile, 20% weight
- Tail: 95th percentile, 10% weight

In [ ]:
quantile_map={'Severe bear':.05,'Bear':.25,'Base':.50,'Bull':.75,'Tail':.95}
weights={'Severe bear':.10,'Bear':.20,'Base':.40,'Bull':.20,'Tail':.10}
states=pd.DataFrame({
    'state':list(quantile_map),
    'quantile':list(quantile_map.values()),
    'terminal_price':[float(targets.target.quantile(q)) for q in quantile_map.values()],
    'subjective_weight':[weights[k] for k in quantile_map]
})
assert np.isclose(states.subjective_weight.sum(),1.0)
states

## 4. Analyst-scenario payoff decomposition

In [ ]:
def payoff(stock,strike,kind):
    stock=np.asarray(stock,float)
    return np.maximum(stock-strike,0) if kind=='call' else np.maximum(strike-stock,0)
states['option_payoff']=payoff(states.terminal_price,M['strike'],M['option_type'])
states['weighted_payoff']=states.subjective_weight*states.option_payoff
weighted_terminal_payoff=float(states.weighted_payoff.sum())
discount_factor=float(np.exp(-M['risk_free_rate']*M['maturity_years']))
belief_value=discount_factor*weighted_terminal_payoff
weight_positive=float(states.loc[states.option_payoff>0,'subjective_weight'].sum())
weight_zero=1-weight_positive
print(f'Weight producing zero payoff: {weight_zero:.1%}')
print(f'Weight producing positive payoff: {weight_positive:.1%}')
print(f'Weighted terminal payoff: ${weighted_terminal_payoff:,.2f}')
print(f'Discount factor: {discount_factor:.4f}')
print(f'Analyst-scenario discounted expected payoff: ${belief_value:,.2f}')
states

### Interpretation
A low analyst-scenario value is intuitive when much of the assigned weight ends below the call strike or above the put strike. The payoff contribution is visible state by state. This is not a replication price.

### 4a. Sensitivity to the state weights
The disclosed weights are a judgment. This table shows how much the scenario value depends on them by repricing the same five states under (i) equal weights and (ii) data-implied weights, where each state receives the empirical share of analyst targets closest to its quantile price.

In [ ]:
def state_value(w: np.ndarray) -> float:
    w=np.asarray(w,float); w=w/w.sum()
    return float(discount_factor*np.sum(w*states.option_payoff.values))

state_prices=states.terminal_price.values
nearest=np.abs(targets.target.values[:,None]-state_prices[None,:]).argmin(axis=1)
data_implied=np.bincount(nearest,minlength=len(states))/len(targets)
weight_scenarios=pd.DataFrame({
    'disclosed':states.subjective_weight.values,'equal':np.full(len(states),1/len(states)),'data_implied':data_implied},index=states.state)
weight_sensitivity={k:state_value(weight_scenarios[k].values) for k in weight_scenarios}
display(weight_scenarios.style.format('{:.1%}'))
pd.Series(weight_sensitivity,name='discounted expected payoff').round(4)

## 5. Distribution-based belief value
Five states give a coarse estimate. Here a Gaussian kernel density (Silverman bandwidth) is fitted to the analyst targets, 20,000 terminal prices are drawn from it with a fixed seed, and the discounted mean payoff is computed. A 2,000-resample bootstrap of the analyst targets gives a confidence interval for this value. Like the five-state value, this is a **belief-weighted** expectation, not a risk-neutral price.

In [ ]:
def kde_sample(x: np.ndarray, n: int, rng: np.random.Generator) -> np.ndarray:
    """Draw from a Gaussian KDE: resample observations, then add bandwidth noise."""
    x=np.asarray(x,float); sd=x.std(ddof=1); iqr=np.subtract(*np.percentile(x,[75,25]))
    h=0.9*min(sd,iqr/1.34)*len(x)**(-1/5)
    draws=rng.choice(x,size=n,replace=True)+rng.normal(0,h,size=n)
    return np.maximum(draws,0.01)

rng=np.random.default_rng(42); N_SIM=20_000
sim_prices=kde_sample(targets.target.values,N_SIM,rng)
sim_payoffs=payoff(sim_prices,M['strike'],M['option_type'])
kde_value=float(discount_factor*sim_payoffs.mean())
kde_prob_itm=float((sim_payoffs>0).mean())
boot=[]
for _ in range(2000):
    xb=rng.choice(targets.target.values,size=len(targets),replace=True)
    boot.append(discount_factor*payoff(kde_sample(xb,2000,rng),M['strike'],M['option_type']).mean())
kde_ci=[float(np.percentile(boot,2.5)),float(np.percentile(boot,97.5))]
hist_counts,hist_edges=np.histogram(sim_prices,bins=40)
print(f'KDE belief value: ${kde_value:,.2f}  (95% bootstrap interval ${kde_ci[0]:,.2f} to ${kde_ci[1]:,.2f})')
print(f'Simulated probability of finishing in the money: {kde_prob_itm:.1%}')
print(f'Five-state disclosed-weight value for comparison: ${belief_value:,.2f}')

## 6. CRR risk-neutral lattice

In [ ]:
def crr(spot,strike,T,r,sigma,N,q=0,kind='call',american=False):
    dt=T/N; u=np.exp(sigma*np.sqrt(dt)); d=1/u; growth=np.exp((r-q)*dt); p=(growth-d)/(u-d); disc=np.exp(-r*dt)
    if not(d<growth<u and 0<p<1): raise ValueError('No-arbitrage condition failed.')
    j=np.arange(N+1); stock=spot*u**j*d**(N-j); values=payoff(stock,strike,kind)
    for n in range(N-1,-1,-1):
        values=disc*(p*values[1:n+2]+(1-p)*values[:n+1])
        if american:
            j=np.arange(n+1); stock=spot*u**j*d**(n-j); values=np.maximum(values,payoff(stock,strike,kind))
    return {'price':float(values[0]),'u':float(u),'d':float(d),'p_star':float(p),'dt':float(dt)}
lattice=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],M['option_type'],M['exercise_style']=='american')
lattice

## 7. Theoretical validation and distribution comparison

In [ ]:
pvs=M['spot']*np.exp(-M['dividend_yield']*M['maturity_years']); pvk=M['strike']*np.exp(-M['risk_free_rate']*M['maturity_years'])
lo,hi=(max(pvs-pvk,0),pvs) if M['option_type']=='call' else (max(pvk-pvs,0),pvk)
call=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],'call')['price']
put=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],'put')['price']
parity=float((call-put)-(pvs-pvk)); steps=[25,50,100,200,500]
conv=[crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],n,M['dividend_yield'],M['option_type'])['price'] for n in steps]
analyst_log_dispersion=float(np.log(targets.target/M['spot']).std(ddof=1))
vol_horizon_dispersion=float(M['volatility']*np.sqrt(M['maturity_years']))
distribution_message='Analyst targets are narrower than the volatility input used by CRR.' if analyst_log_dispersion<vol_horizon_dispersion else 'Analyst targets are at least as dispersed as the volatility input used by CRR.'
validation={'valid_risk_neutral_probability':bool(0<lattice['p_star']<1),'inside_theoretical_bounds':bool(lo<=lattice['price']<=hi),'put_call_parity_pass':bool(abs(parity)<.05),'convergence_pass':bool(abs(conv[-1]-conv[-2])<.05),'parity_residual':parity,'convergence_gap':float(conv[-1]-conv[-2])}
assert all(validation[k] for k in ['valid_risk_neutral_probability','inside_theoretical_bounds','put_call_parity_pass','convergence_pass'])
print(distribution_message)
validation

### 7a. Black-Scholes-Merton benchmark
For a European option the CRR lattice should converge to the closed-form Black-Scholes-Merton value with a continuous dividend yield. The gap is an independent check on the lattice. (For American exercise the closed form is a lower bound for calls on dividend payers and is reported for reference only.)

In [ ]:
_ncdf=np.vectorize(lambda z: 0.5*(1+erf(z/sqrt(2))))
def black_scholes(spot: float, strike: float, T: float, r: float, sigma: float, q: float=0.0, kind: str='call') -> dict:
    d1=(np.log(spot/strike)+(r-q+0.5*sigma**2)*T)/(sigma*np.sqrt(T)); d2=d1-sigma*np.sqrt(T)
    if kind=='call':
        price=spot*np.exp(-q*T)*_ncdf(d1)-strike*np.exp(-r*T)*_ncdf(d2); delta=np.exp(-q*T)*_ncdf(d1)
    else:
        price=strike*np.exp(-r*T)*_ncdf(-d2)-spot*np.exp(-q*T)*_ncdf(-d1); delta=-np.exp(-q*T)*_ncdf(-d1)
    vega=spot*np.exp(-q*T)*np.exp(-d1**2/2)/np.sqrt(2*np.pi)*np.sqrt(T)
    return {'price':float(price),'d1':float(d1),'d2':float(d2),'delta':float(delta),'vega_per_vol_point':float(vega/100),'risk_neutral_prob_itm':float(_ncdf(d2) if kind=='call' else _ncdf(-d2))}

bsm=black_scholes(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['dividend_yield'],M['option_type'])
bsm['crr_minus_bsm']=lattice['price']-bsm['price']
bsm['benchmark_pass']=bool(M['exercise_style']=='american' or abs(bsm['crr_minus_bsm'])<0.05)
bsm

### 7b. Market comparison and implied volatility
The model values are compared with the quoted bid, midpoint and ask. The implied volatility that reprices the market midpoint (bisection on Black-Scholes-Merton) shows what volatility the market quote embeds.

In [ ]:
def implied_vol(target_price: float, lo: float=1e-4, hi: float=3.0, tol: float=1e-8) -> float|None:
    f=lambda s: black_scholes(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],s,M['dividend_yield'],M['option_type'])['price']-target_price
    if f(lo)*f(hi)>0: return None
    for _ in range(200):
        mid=(lo+hi)/2
        if f(lo)*f(mid)<=0: hi=mid
        else: lo=mid
        if hi-lo<tol: break
    return float((lo+hi)/2)

spread=M['market_ask']-M['market_bid']
mispricing={
    'market_mid_minus_crr':M['market_mid']-lattice['price'],
    'market_mid_vs_crr_pct':M['market_mid']/lattice['price']-1,
    'crr_inside_bid_ask':bool(M['market_bid']<=lattice['price']<=M['market_ask']),
    'bid_ask_spread':spread,'spread_pct_of_mid':spread/M['market_mid'],
    'market_mid_minus_belief_value':M['market_mid']-belief_value,
    'market_mid_minus_kde_value':M['market_mid']-kde_value,
    'implied_vol_bid':implied_vol(M['market_bid']),'implied_vol_mid':implied_vol(M['market_mid']),'implied_vol_ask':implied_vol(M['market_ask']),
}
mispricing['implied_minus_input_vol']=(mispricing['implied_vol_mid']-M['volatility']) if mispricing['implied_vol_mid'] else None
pd.Series(mispricing)

## 8. Sensitivity analysis
Each row reprices the option with the CRR lattice, changing one input at a time and holding the rest at base values.

In [ ]:
def price_with(**over) -> float:
    p={'spot':M['spot'],'strike':M['strike'],'T':M['maturity_years'],'r':M['risk_free_rate'],'sigma':M['volatility'],'q':M['dividend_yield'],**over}
    return crr(p['spot'],p['strike'],p['T'],p['r'],p['sigma'],M['lattice_steps'],p['q'],M['option_type'],M['exercise_style']=='american')['price']

def grid(base: float, rel: list[float]) -> list[float]:
    return [base*(1+x) for x in rel]

sensitivity={
 'volatility':[{'input':s,'price':price_with(sigma=s)} for s in [M['volatility']+x for x in (-.10,-.05,0,.05,.10) if M['volatility']+x>0]],
 'risk_free_rate':[{'input':r,'price':price_with(r=r)} for r in [max(M['risk_free_rate']+x,0) for x in (-.02,-.01,0,.01,.02)]],
 'strike':[{'input':k,'price':price_with(strike=k)} for k in grid(M['strike'],[-.10,-.05,0,.05,.10])],
 'maturity_years':[{'input':t,'price':price_with(T=t)} for t in grid(M['maturity_years'],[-.5,-.25,0,.25,.5])],
 'spot':[{'input':s,'price':price_with(spot=s)} for s in grid(M['spot'],[-.10,-.05,0,.05,.10])],
}
for k,v in sensitivity.items():
    print(k); display(pd.DataFrame(v).round(4))

## 9. Maturity alignment, source audit and model-risk flags

In [ ]:
analyst_horizon_years=float(targets.horizon_months.iloc[0])/12
alignment_gap=abs(analyst_horizon_years-M['maturity_years'])
maturity_alignment={'analyst_horizon_years':analyst_horizon_years,'option_maturity_years':M['maturity_years'],'alignment_gap_years':alignment_gap,
    'aligned':bool(alignment_gap<=0.25),
    'message':'Analyst horizon and option maturity are aligned.' if alignment_gap<=0.25 else f'Warning: analyst horizon differs from option maturity by {alignment_gap:.2f} years; state payoffs mix horizons.'}
print(maturity_alignment['message'])

src=pd.read_csv(DATA/'source_documentation.csv')
for col in ['source_url','retrieval_date']:
    if col not in src.columns: src[col]=''
src=src.fillna('')
source_audit={'documented_inputs':int(len(src)),
    'synthetic_inputs':[n for n,s in zip(src.input_name,src.source) if 'synthetic' in s.lower()],
    'missing_url':[n for n,u,s in zip(src.input_name,src.source_url,src.source) if not str(u).strip() and 'student model' not in s.lower()],
    'analyst_targets_synthetic':bool(targets.source_name.str.contains('synthetic',case=False).any())}
source_audit['complete']=not(source_audit['synthetic_inputs'] or source_audit['missing_url'] or source_audit['analyst_targets_synthetic'])

def flag(name: str, ok: bool, detail: str) -> dict: return {'check':name,'status':'pass' if ok else 'warn','detail':detail}
model_risk=[
 flag('Horizon alignment',maturity_alignment['aligned'],maturity_alignment['message']),
 flag('Put-call parity',validation['put_call_parity_pass'],f"Residual {validation['parity_residual']:.2e}"),
 flag('Lattice convergence',validation['convergence_pass'],f"200 vs 500 step gap {validation['convergence_gap']:+.4f}"),
 flag('Black-Scholes benchmark',bsm['benchmark_pass'],f"CRR minus BSM {bsm['crr_minus_bsm']:+.4f}"),
 flag('Analyst vs. option volatility',analyst_log_dispersion>=vol_horizon_dispersion,f"Analyst log dispersion {analyst_log_dispersion:.1%} vs CRR horizon volatility {vol_horizon_dispersion:.1%}"),
 flag('Scenario-weight robustness',max(weight_sensitivity.values())-min(weight_sensitivity.values())<0.25*max(abs(belief_value),1e-9),
      f"Five-state value ranges ${min(weight_sensitivity.values()):.2f} to ${max(weight_sensitivity.values()):.2f} across weighting schemes"),
 flag('Analyst outliers',len(diagnostics['tukey_outliers'])==0,f"Tukey outliers: {diagnostics['tukey_outliers'] or 'none'}"),
 flag('Bid-ask liquidity',mispricing['spread_pct_of_mid']<0.10,f"Spread is {mispricing['spread_pct_of_mid']:.1%} of midpoint"),
 flag('Source documentation',source_audit['complete'],'All inputs sourced with URLs' if source_audit['complete'] else 'Synthetic or unsourced inputs remain; replace before submission'),
]
if dcf: model_risk.append(flag('Terminal-value concentration',dcf['terminal_share_of_ev']<0.75,f"Terminal value is {dcf['terminal_share_of_ev']:.0%} of DCF enterprise value"))
pd.DataFrame(model_risk)

## 10. Export dashboard data
The original keys used by `docs/index.html` are unchanged. New results are added under `extended` for `docs/analysis.html`, and a dated input snapshot is written to `docs/snapshots/` for reproducibility.

In [ ]:
results={
 'model':M,'analyst_targets':[float(x) for x in targets.target],'analyst_stats':stats,'valuation':valuation,
 'states':states.to_dict('records'),'valuation_weighted_expected_payoff':belief_value,
 'belief_decomposition':{'weight_zero_payoff':weight_zero,'weight_positive_payoff':weight_positive,'weighted_terminal_payoff':weighted_terminal_payoff,'discount_factor':discount_factor},
 'distribution_comparison':{'analyst_log_dispersion':analyst_log_dispersion,'crr_horizon_volatility':vol_horizon_dispersion,'message':distribution_message},
 'lattice':lattice,'validation':validation,'convergence':[{'steps':n,'price':float(x)} for n,x in zip(steps,conv)],
 'comparison':{'Analyst-scenario discounted expected payoff':belief_value,'CRR risk-neutral lattice value':lattice['price'],'Observed market midpoint':M['market_mid']}
}
results['extended']={
 'valuation_upside':{k:float(v) for k,v in valuation_table.upside_vs_spot.items()},
 'dcf':None if dcf is None else {**{k:float(v) for k,v in dcf.items() if k!='projection'},'projection':dcf['projection'].to_dict('records'),
        'sensitivity':{c:dict(zip(dcf_sensitivity.index,map(float,dcf_sensitivity[c]))) for c in dcf_sensitivity}},
 'distribution_diagnostics':diagnostics,
 'weight_sensitivity':weight_sensitivity,'data_implied_weights':dict(zip(states.state,map(float,data_implied))),
 'kde':{'value':kde_value,'bootstrap_95ci':kde_ci,'prob_itm':kde_prob_itm,'n_sim':N_SIM,'seed':42,
        'histogram':{'counts':hist_counts.tolist(),'edges':hist_edges.tolist()}},
 'black_scholes':bsm,'mispricing':mispricing,'sensitivity':sensitivity,
 'maturity_alignment':maturity_alignment,'source_audit':source_audit,'model_risk':model_risk,
 'volatility_card':{'input':M['volatility'],'type':str(M.get('volatility_type','')),'implied_from_mid':mispricing['implied_vol_mid'],
        'analyst_log_dispersion':analyst_log_dispersion,'vega_per_vol_point':bsm['vega_per_vol_point']},
}
(DOCS/'model_results.json').write_text(json.dumps(results,indent=2,default=float))
snap=DOCS/'snapshots'; snap.mkdir(exist_ok=True)
inputs.to_csv(snap/f"company_inputs-{M['valuation_date']}.csv",index=False)
targets.to_csv(snap/f"analyst_targets-{M['valuation_date']}.csv",index=False)
print('Exported docs/model_results.json and dated input snapshots')
results['comparison']

### Evidence summary (generated from the results)
This cell prints the numbers the memo should cite, so the memo stays consistent with the model when inputs change.

In [ ]:
print(f"Spot ${M['spot']:.2f}; DCF ${M['dcf_value']:.2f} ({valuation_table.upside_vs_spot['Student DCF']:+.1%}); comps ${M['comps_value']:.2f}; analyst median ${stats['median']:.2f}.")
print(f"Analyst targets: n={stats['count']}, mean ${stats['mean']:.2f}, sd ${stats['std']:.2f}, skew {diagnostics['skewness']:.2f}, IQR ${diagnostics['iqr']:.2f}, outliers {diagnostics['tukey_outliers'] or 'none'}.")
print(f"{weight_zero:.0%} of disclosed scenario weight finishes out of the money; five-state value ${belief_value:.2f} (equal weights ${weight_sensitivity['equal']:.2f}, data-implied ${weight_sensitivity['data_implied']:.2f}); KDE value ${kde_value:.2f}.")
print(f"CRR ${lattice['price']:.2f} vs Black-Scholes ${bsm['price']:.2f}; market bid/mid/ask ${M['market_bid']:.2f}/${M['market_mid']:.2f}/${M['market_ask']:.2f}; mid vs CRR {mispricing['market_mid_vs_crr_pct']:+.1%}.")
print(f"Implied vol at mid {mispricing['implied_vol_mid']:.1%} vs input {M['volatility']:.1%}; analyst log dispersion {analyst_log_dispersion:.1%}.")
print(maturity_alignment['message'])

## 11. Decision memo

Explain why the analyst-scenario payoff and CRR value differ. Address the share of scenario weight producing a zero payoff, the contribution from positive-payoff states, the strike, analyst-target dispersion, the CRR volatility input, and why neither estimate should automatically be characterized as the uniquely correct market price.

Use the evidence summary above. Cover each required section:

### Equity valuation
*Current price, DCF (including the explicit build and its terminal-value share), comparable value, and analyst median.*

### Analyst distribution
*Central tendency, dispersion, skew, outliers, dependence between analysts, and limitations.*

### Option pricing
*Five-state belief value and its weight sensitivity, the KDE value and interval, CRR value, Black-Scholes benchmark, bid, midpoint and ask.*

### Model risk
*Maturity alignment, volatility choice (input vs implied vs analyst dispersion), dividends, liquidity and spread, state weights, and model specification.*

### Conclusion
*What the evidence supports, without automatically calling the option mispriced.*